POR MAS QUE LLAME A OPERACIONES_RAW, EN UN ENTORNO REAL DIRECTAMENTE CREARIA UN PIPELINE PARA QUE SEA ACTUALICE LA TABLA MEDIANTE BATCH

In [0]:
%run ../ingesta/operaciones_raw

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

INPUT_PATH = "/Volumes/platform_dev/iol_challenge/input/*.csv"
tabla_origen = "platform_dev.bronce_byma.operaciones_raw"
tabla_destino = 'platform_dev.silver_byma.operaciones_replica'

dbutils.widgets.text("full_load", "0")

full_load = int(dbutils.widgets.get("full_load"))

carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("operaciones_replica")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(
            f"La tabla {tabla_destino} no existe. "
            "Se ejecutará el notebook de creación de tablas."
        )

        dbutils.notebook.run("../DDL/creacion_tablas", 0)

        carga_full = True

        logger.info(
            "Tablas y esquemas creados correctamente. "
            "Se realizará una carga full."
        )

    elif carga_full:
        logger.warning(
            f"Carga full forzada manualmente para {tabla_destino}."
        )

    else:
        logger.info(f"La tabla {tabla_destino} existe.")

        tiene_datos = spark.sql(
            f"SELECT 1 FROM {tabla_destino} LIMIT 1"
        ).take(1) != []

        if tiene_datos:
            logger.info(
                f"La tabla {tabla_destino} contiene datos. "
                "Se realizará una carga incremental."
            )
        else:
            carga_full = True

            logger.warning(
                f"La tabla {tabla_destino} está vacía. "
                "Se realizará una carga full."
            )

except Exception:
    logger.exception(
        f"Error al validar o crear la tabla {tabla_destino}"
    )
    raise

In [0]:
# 1. Resolución segura del Watermark evitando IndexErrors
if not carga_full:
    fila_control = (
        spark.table("platform_dev.bronce_byma.control_cargas")
        .filter(F.col("proceso") == tabla_origen)
        .select("ultima_fecha_procesada")
        .first()
    )
    
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
        logger.info(f"Carga incremental activada. Leyendo desde: {ultima_fecha}")
    else:
        ultima_fecha = "1970-01-01"
        logger.warning(f"No se encontró registro para {tabla_origen}. Procesando desde base: {ultima_fecha}")
else:
    ultima_fecha = "1970-01-01"

# 2. Bloque de extracción protegido con Exception as e
try:
    if carga_full:
        df = spark.table(f"{tabla_origen}")
        logger.info(f"Ejecutando Carga histórica de {tabla_origen}")
    else:
        df = spark.table(f"{tabla_origen}").filter(F.col("fecha_particion") >= F.lit(ultima_fecha))
        logger.info(f"Ejecutando Carga filtrada de {tabla_origen} desde {ultima_fecha}")
except Exception as e:
    logger.error(f"Error crítico al leer la tabla {tabla_origen}: {str(e)}")
    raise

# 3. Transformaciones y Normalización de Zona Horaria
df_replica = df.withColumn("fecha_utc", F.to_timestamp("fecha")) \
               .withColumn("fecha_local", F.from_utc_timestamp(F.col("fecha_utc"), "America/Argentina/Buenos_Aires")) \
               .withColumn("cantidad", F.col("cantidad").cast("decimal(18,4)")) \
               .withColumn("precio", F.col("precio").cast("decimal(18,4)")) \
               .withColumn("_processed_at", F.current_timestamp())

# 4. Ventana de Deduplicación por id_transaccion
window = Window.partitionBy("id_transaccion").orderBy(F.col("_ingested_at").desc())

df_merge = (
    df_replica
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

df_merge.createOrReplaceTempView("operaciones_replica_tmp")

In [0]:
cantidad_registros = df_merge.count()
try:
    if carga_full:
        logger.info(
        f"Carga full: iniciando escritura de {cantidad_registros} registros "
        f"en {tabla_destino}")

        df_merge.write \
            .format("delta") \
                .mode("overwrite") \
                    .saveAsTable(tabla_destino)

        logger.info(
            f"Carga full finalizada: {cantidad_registros} registros insertados")

    else:
        if cantidad_registros == 0:
            logger.info("Carga incremental: no hay registros nuevos para insertar")

        else:
            logger.info(
                f"Carga incremental: iniciando escritura de "
                f"{cantidad_registros} registros en {tabla_destino}"
            )

            spark.sql(f"""
                MERGE INTO {tabla_destino} AS target
                USING operaciones_replica_tmp AS source
                ON target.id_transaccion = source.id_transaccion

                WHEN MATCHED THEN UPDATE SET *

                WHEN NOT MATCHED THEN INSERT *
                """)

            logger.info(
                f"Carga incremental finalizada: "
                f"{cantidad_registros} registros insertados"
            )

except Exception:
    logger.exception(f"Error durante la escritura en {tabla_destino}")
    raise